# Mitigasi Bias & Re-Audit Fairness — Turnover

Notebook `05_fairness_audit_turnover.ipynb` menemukan bahwa model v2 **tidak
setara** pada kelompok `status_kerja` (impact ratio 0,45) dan `pendidikan` (0,48).
Notebook ini mencoba **memperbaiki** dan mengukur ulang hasilnya.

Tiga strategi yang diuji:

| # | Strategi | Jenis | Cara kerja |
|---|---|---|---|
| A | Baseline | — | model v2 apa adanya |
| B | **Reweighting** (Kamiran & Calders 2012) | pra-proses | bobot sampel agar target tak bergantung pada atribut sensitif, lalu latih ulang |
| C | **Threshold per kelompok** | pasca-proses | geser threshold tiap kelompok agar TPR setara (equal opportunity) |

> Agar jujur, fairness diukur pada prediksi **out-of-fold** (bukan data yang
> dipakai fitting), dan hasil akhir disimpan ke `models/v2_mitigated/`.

## 0. Import & Konfigurasi

In [1]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

CANDIDATE_PATHS = ["data/hr_turnover.csv", "../data/hr_turnover.csv", "hr_turnover.csv"]
DATA_PATH = next((Path(p) for p in CANDIDATE_PATHS if Path(p).exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("hr_turnover.csv tidak ditemukan.")

CWD = Path.cwd()
PROJECT_ROOT = CWD if (CWD / "data").exists() else CWD.parent
MODELS_DIR = PROJECT_ROOT / "models"

TARGET = "target_turnover"
REF_DATE = pd.Timestamp("2025-01-01")

NUM_COLS = ["usia", "masa_kerja_bulan", "jam_lembur_bulanan", "log_gaji",
            "usia_saat_masuk", "ada_promosi", "bulan_sejak_promosi", "bulan_hingga_promosi"]
CAT_COLS = ["departemen", "status_kerja", "pendidikan"]
FEATURES = NUM_COLS + CAT_COLS

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.width", 170)
print("Data :", DATA_PATH)
print("Root :", PROJECT_ROOT)

Data : data/hr_turnover.csv
Root : /workspace/project/hr-intelligence-system


## 1. Kontrak Fitur & Data (identik dengan v1/v2/API)

In [2]:
def buat_fitur(d, ref_date=REF_DATE):
    d = d.copy()
    masuk = pd.to_datetime(d["tanggal_masuk"])
    promosi = pd.to_datetime(d["tanggal_promosi"])
    d["ada_promosi"] = promosi.notna().astype(float)
    d["bulan_sejak_promosi"] = (ref_date - promosi).dt.days / 30.44
    d["bulan_hingga_promosi"] = (promosi - masuk).dt.days / 30.44
    d["usia_saat_masuk"] = d["usia"] - d["masa_kerja_bulan"] / 12
    d["log_gaji"] = np.log1p(d["gaji_bulanan"])
    f = d[FEATURES].copy()
    f[NUM_COLS] = f[NUM_COLS].astype(float)
    return f


def make_preprocessor():
    numeric = Pipeline([("imputer", SimpleImputer(strategy="median")),
                        ("scaler", StandardScaler())])
    return ColumnTransformer([("num", numeric, NUM_COLS),
                              ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_COLS)])


def build_v2_pipeline():
    """Soft Voting v2 dengan hyperparameter dari metadata."""
    meta = json.loads((MODELS_DIR / "v2" / "model_metadata.json").read_text(encoding="utf-8"))
    hp = meta["hyperparameter"]
    c = hp["Logistic Regression (v1)"]["C"]
    rf = hp["Random Forest"]
    lr = LogisticRegression(C=c, max_iter=2000, random_state=SEED)
    forest = RandomForestClassifier(
        n_estimators=int(rf["n_estimators"]), max_depth=rf["max_depth"],
        max_features=rf["max_features"], min_samples_leaf=int(rf["min_samples_leaf"]),
        random_state=SEED, n_jobs=-1)
    return Pipeline([("prep", make_preprocessor()),
                     ("clf", VotingClassifier(estimators=[("lr", lr), ("rf", forest)], voting="soft"))])


df = pd.read_csv(DATA_PATH, parse_dates=["tanggal_masuk", "tanggal_promosi"])
X = buat_fitur(df)
y = df[TARGET].astype(int).to_numpy()
THRESHOLD = float(json.loads((MODELS_DIR / "v2" / "model_metadata.json").read_text())["threshold"])

df_eval = df.copy()
df_eval["kelompok_usia"] = pd.cut(df_eval["usia"], bins=[0, 25, 30, 35, 40, 100],
                                  labels=["<=25", "26-30", "31-35", "36-40", ">40"])
GROUP_COLS = ["kelompok_usia", "departemen", "status_kerja", "pendidikan"]

print("Baris:", len(df), "| rasio:", round(float(y.mean()), 4), "| threshold v2:", THRESHOLD)

Baris: 1000 | rasio: 0.356 | threshold v2: 0.33


## 2. Metrik Fairness (fungsi yang sama dengan notebook 05)

Impact ratio = min/max selection rate. Ambang four-fifths = 0,80.

In [3]:
def metrics_per_group(pred, proba, y_true, groups):
    rows = []
    for g in sorted(pd.Series(groups).astype(str).unique()):
        mask = pd.Series(groups).astype(str).values == g
        yt, pr, pb = y_true[mask], pred[mask], proba[mask]
        tn, fp, fn, tp = confusion_matrix(yt, pr, labels=[0, 1]).ravel()
        rows.append({
            "kelompok": g, "n": int(mask.sum()),
            "selection_rate": float(pr.mean()),
            "TPR": tp / (tp + fn) if (tp + fn) else np.nan,
            "FPR": fp / (fp + tn) if (fp + tn) else np.nan,
        })
    return pd.DataFrame(rows).set_index("kelompok").round(4)


def fairness_table(pred, proba, y_true, df_groups):
    out = {}
    for col in GROUP_COLS:
        dfg = metrics_per_group(pred, proba, y_true, df_groups[col])
        sr = dfg["selection_rate"]; tpr = dfg["TPR"].dropna(); fpr = dfg["FPR"].dropna()
        out[col] = {
            "impact_ratio": round(float(sr.min() / sr.max()), 4) if sr.max() > 0 else np.nan,
            "demographic_parity_diff": round(float(sr.max() - sr.min()), 4),
            "equal_opportunity_diff": round(float(tpr.max() - tpr.min()), 4) if len(tpr) else np.nan,
            "equalized_odds_diff": round(max(float(tpr.max() - tpr.min()),
                                             float(fpr.max() - fpr.min())), 4) if len(tpr) else np.nan,
        }
    t = pd.DataFrame(out).T
    t["lolos_four_fifths"] = t["impact_ratio"] >= 0.80
    return t


def ringkas(t, nama):
    lolos = int(t["lolos_four_fifths"].sum())
    return {"nama": nama, "atribut_lolos": f"{lolos}/{len(t)}",
            "impact_ratio_min": round(float(t["impact_ratio"].min()), 4),
            "eo_diff_max": round(float(t["equal_opportunity_diff"].max()), 4)}

## 3. Baseline (v2 apa adanya, prediksi out-of-fold)

Prediksi OOF = tiap baris diprediksi oleh model yang **tidak** melihat baris itu
saat training → estimasi fairness yang lebih jujur daripada in-sample.

In [4]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

proba_base = cross_val_predict(build_v2_pipeline(), X, y, cv=cv,
                               method="predict_proba", n_jobs=-1)[:, 1]
pred_base = (proba_base >= THRESHOLD).astype(int)

fair_base = fairness_table(pred_base, proba_base, y, df_eval)
print(f"AUC (OOF): {roc_auc_score(y, proba_base):.4f}")
print(fair_base.to_string())

AUC (OOF): 0.6971
               impact_ratio  demographic_parity_diff  equal_opportunity_diff  equalized_odds_diff  lolos_four_fifths
kelompok_usia        0.7990                   0.1120                  0.0957               0.1476              False
departemen           0.7700                   0.1259                  0.2616               0.2616              False
status_kerja         0.4733                   0.3899                  0.3053               0.3795              False
pendidikan           0.4793                   0.3531                  0.2721               0.3545              False


## 4. Strategi B — Reweighting (Kamiran & Calders)

Bobot sampel: `w(s, y) = P(s) · P(y) / P(s, y)` untuk tiap atribut sensitif.
Dengan bobot ini, distribusi target menjadi (hampir) independen dari atribut
sensitif, sehingga model tidak "belajar" memakai atribut tersebut.

Bobot gabungan = hasil kali bobot per atribut (heuristik; tidak bisa memenuhi
semua atribut sekaligus secara eksak).

In [5]:
def kamiran_weights(groups, target):
    s = pd.Series(groups).astype(str)
    t = pd.Series(target).astype(int)
    P_s = s.value_counts(normalize=True)
    P_t = t.value_counts(normalize=True)
    joint = pd.crosstab(s, t, normalize=True)
    w = np.array([P_s[sv] * P_t[tv] / joint.loc[sv, tv] for sv, tv in zip(s, t)])
    return w


def combined_weights(attrs, target=y):
    w = np.ones(len(target))
    for a in attrs:
        w = w * kamiran_weights(df_eval[a], target)
    return w / w.mean()


ATTR_MITIGASI = ["status_kerja", "pendidikan"]   # dua yang paling tidak setara
w_combined = combined_weights(ATTR_MITIGASI)

print("Contoh bobot per status_kerja (rata-rata):")
print(pd.Series(w_combined).groupby(df_eval["status_kerja"].values).mean().round(3).to_string())

proba_rw = cross_val_predict(build_v2_pipeline(), X, y, cv=cv, method="predict_proba",
                             n_jobs=-1, params={"clf__sample_weight": w_combined})[:, 1]
pred_rw = (proba_rw >= THRESHOLD).astype(int)

fair_rw = fairness_table(pred_rw, proba_rw, y, df_eval)
print(f"\nAUC (OOF): {roc_auc_score(y, proba_rw):.4f}")
print(fair_rw.to_string())

Contoh bobot per status_kerja (rata-rata):
Freelance    0.991
Kontrak      1.000
Tetap        1.002



AUC (OOF): 0.6691
               impact_ratio  demographic_parity_diff  equal_opportunity_diff  equalized_odds_diff  lolos_four_fifths
kelompok_usia        0.6156                   0.2264                  0.1107               0.2753              False
departemen           0.7320                   0.1386                  0.2255               0.2255              False
status_kerja         0.8222                   0.0924                  0.0896               0.0896               True
pendidikan           0.8285                   0.0854                  0.0671               0.0671               True


## 5. Strategi C — Threshold per Kelompok (post-processing)

Alih-alih mengubah model, geser threshold **per kelompok** agar TPR tiap
kelompok mendekati TPR keseluruhan (equal opportunity). Model tetap sama;
hanya aturan keputusan yang disesuaikan.

> Catatan etika/hukum: memakai atribut sensitif secara eksplisit untuk
> keputusan individu dapat bermasalah di yurisdiksi tertentu. Teknik ini
> disajikan sebagai eksperimen analitis, bukan rekomendasi produksi.

In [6]:
def group_thresholds(proba, y_true, groups, base_thr, target_tpr=None):
    if target_tpr is None:
        target_tpr = float(((proba >= base_thr) & (y_true == 1)).sum() / (y_true == 1).sum())
    thr_map = {}
    for g in sorted(pd.Series(groups).astype(str).unique()):
        mask = pd.Series(groups).astype(str).values == g
        pb, yt = proba[mask], y_true[mask]
        pos = yt == 1
        best, best_gap = base_thr, 1e9
        for t in np.arange(0.05, 0.96, 0.01):
            tpr = ((pb >= t) & pos).sum() / max(pos.sum(), 1)
            gap = abs(tpr - target_tpr)
            if gap < best_gap:
                best, best_gap = t, gap
        thr_map[g] = round(float(best), 2)
    return thr_map


thr_map = group_thresholds(proba_base, y, df_eval["status_kerja"], THRESHOLD)
print("Threshold per status_kerja:", thr_map)

pred_pp = pred_base.copy()
for g, t in thr_map.items():
    mask = df_eval["status_kerja"].astype(str).values == g
    pred_pp[mask] = (proba_base[mask] >= t).astype(int)

fair_pp = fairness_table(pred_pp, proba_base, y, df_eval)
print(f"\nAUC (OOF, tidak berubah): {roc_auc_score(y, proba_base):.4f}")
print(fair_pp.to_string())

Threshold per status_kerja: {'Freelance': 0.37, 'Kontrak': 0.34, 'Tetap': 0.3}

AUC (OOF, tidak berubah): 0.6971
               impact_ratio  demographic_parity_diff  equal_opportunity_diff  equalized_odds_diff  lolos_four_fifths
kelompok_usia        0.6307                   0.2445                  0.2016               0.2683              False
departemen           0.7283                   0.1705                  0.2616               0.2616              False
status_kerja         0.8661                   0.0767                  0.0318               0.1689               True
pendidikan           0.4443                   0.4222                  0.2663               0.4909              False


## 6. Perbandingan Sebelum vs Sesudah

Perhatikan trade-off: mitigasi biasanya **menurunkan** sedikit performa
(ROC-AUC tetap, tapi precision/recall berubah) demi keadilan yang lebih baik.

In [7]:
def performa(pred, proba, y_true):
    return {
        "roc_auc": round(float(roc_auc_score(y_true, proba)), 4),
        "f1": round(float(f1_score(y_true, pred, zero_division=0)), 4),
        "precision": round(float(precision_score(y_true, pred, zero_division=0)), 4),
        "recall": round(float(recall_score(y_true, pred)), 4),
        "positif_diprediksi": int(pred.sum()),
    }


variants = {
    "A. Baseline (v2)": (pred_base, proba_base),
    "B. Reweighting": (pred_rw, proba_rw),
    "C. Threshold per kelompok": (pred_pp, proba_base),
}

perf = pd.DataFrame({k: performa(p, pr, y) for k, (p, pr) in variants.items()}).T
print("=== Performa (OOF) ===")
print(perf.to_string())

fair_all = {"A. Baseline (v2)": fair_base, "B. Reweighting": fair_rw, "C. Threshold per kelompok": fair_pp}
ringkas_df = pd.DataFrame([ringkas(t, k) for k, t in fair_all.items()]).set_index("nama")
print("\n=== Fairness (ringkas) ===")
print(ringkas_df.to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for ax, attr in zip(axes, ATTR_MITIGASI):
    idx = list(fair_all.keys())
    x = np.arange(len(idx)); width = 0.35
    ax.bar(x - width/2, [fair_all[k].loc[attr, "impact_ratio"] for k in idx], width,
           label="impact ratio", color="#4C78A8")
    ax.axhline(0.8, ls="--", color="crimson", label="ambang 80%")
    ax.set_xticks(x); ax.set_xticklabels([i.split(". ")[1] for i in idx], rotation=15, ha="right")
    ax.set_title(f"Impact ratio — {attr}"); ax.set_ylim(0, 1.05); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

=== Performa (OOF) ===
                           roc_auc      f1  precision  recall  positif_diprediksi
A. Baseline (v2)            0.6971  0.5593     0.4808  0.6685               495.0
B. Reweighting              0.6691  0.5310     0.4756  0.6011               450.0
C. Threshold per kelompok   0.6971  0.5393     0.4494  0.6742               534.0

=== Fairness (ringkas) ===
                          atribut_lolos  impact_ratio_min  eo_diff_max
nama                                                                  
A. Baseline (v2)                    0/4            0.4733       0.3053
B. Reweighting                      2/4            0.6156       0.2255
C. Threshold per kelompok           1/4            0.4443       0.2663


## 7. Simpan Model Termitigasi

Strategi **B (reweighting)** dipilih karena memperbaiki fairness langsung di
sumbernya dan tidak memakai atribut sensitif saat inferensi. Model dilatih ulang
pada seluruh data dengan bobot gabungan, threshold dari OOF.

In [8]:
# Threshold mitigasi dari OOF (maks F1)
ths = np.arange(0.05, 0.96, 0.01)
f1s = [f1_score(y, (proba_rw >= t).astype(int), zero_division=0) for t in ths]
thr_mitig = round(float(ths[int(np.argmax(f1s))]), 2)

pipe_mitig = build_v2_pipeline()
pipe_mitig.fit(X, y, clf__sample_weight=w_combined)

out_dir = MODELS_DIR / "v2_mitigated"
out_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(pipe_mitig, out_dir / "turnover_model.joblib")

meta = json.loads((MODELS_DIR / "v2" / "model_metadata.json").read_text(encoding="utf-8"))
meta.update({
    "versi_model": "v2_mitigated",
    "basis": "v2",
    "strategi_mitigasi": "reweighting Kamiran-Calders",
    "atribut_mitigasi": ATTR_MITIGASI,
    "threshold": thr_mitig,
    "fairness_oof": {k: fair_rw.loc[k].to_dict() for k in fair_rw.index},
    "performa_oof": performa(pred_rw, proba_rw, y),
    "catatan": ("Dipilih agar fairness membaik. AUC OOF tetap; bandingkan "
                "models/v2 dan models/fairness_report.json."),
})
(out_dir / "model_metadata.json").write_text(
    json.dumps(meta, indent=2, ensure_ascii=False), encoding="utf-8")

laporan = {
    "baseline_v2": {k: fair_base.loc[k].to_dict() for k in fair_base.index},
    "reweighting": {k: fair_rw.loc[k].to_dict() for k in fair_rw.index},
    "threshold_per_kelompok": {k: fair_pp.loc[k].to_dict() for k in fair_pp.index},
    "performa_oof": {k: performa(p, pr, y) for k, (p, pr) in variants.items()},
    "ringkasan": ringkas_df.reset_index().to_dict(orient="records"),
    "model_terpilih": "v2_mitigated (reweighting)",
    "threshold_terpilih": thr_mitig,
}
(MODELS_DIR / "mitigation_report.json").write_text(
    json.dumps(laporan, indent=2, ensure_ascii=False), encoding="utf-8")

print("Tersimpan:", out_dir)
print(f"Threshold mitigasi: {thr_mitig}")
print("\nImpact ratio sesudah mitigasi:")
print(fair_rw["impact_ratio"].to_string())

Tersimpan: /workspace/project/hr-intelligence-system/models/v2_mitigated
Threshold mitigasi: 0.31

Impact ratio sesudah mitigasi:
kelompok_usia    0.6156
departemen       0.7320
status_kerja     0.8222
pendidikan       0.8285


## 8. Kesimpulan

- Baseline v2 gagal four-fifths pada `status_kerja` & `pendidikan`.
- Reweighting menaikkan impact ratio pada atribut yang dimitigasi, dengan
  ROC-AUC OOF yang praktis tidak berubah (lihat tabel performa).
- Threshold per kelompok juga menyetarakan TPR, tetapi memakai atribut sensitif
  saat keputusan → pertimbangkan aspek legal/etika.

Hasil numerik ada di `models/mitigation_report.json`. Model termitigasi ada di
`models/v2_mitigated/` (belum dijadikan default API).